# World models de conducción sobre tokens de commaVQ (Colab)
Notebook para correr el proyecto en Colab de arriba a abajo. Los pasos largos (entrenar, rollouts, métricas) están comentados: descomentá lo que necesites. Usar GPU: *Entorno de ejecución > Cambiar tipo > T4*. La T4 es la GPU de referencia para las latencias (D1).

## 1. Preparación

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CK = '/content/drive/MyDrive/ml2/checkpoints'  # los checkpoints viven en Drive: Colab corta las sesiones
!mkdir -p {CK}

In [ ]:
%cd /content
!git clone https://github.com/bpeco/ml2-gen-world-model.git
%cd /content/ml2-gen-world-model
!git clone --depth 1 https://github.com/commaai/commavq.git third_party/commavq

In [ ]:
!pip install -q -r requirements.txt -r requirements-metrics.txt
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

### Datos
Shard 0 de train, val oficial y los pesos del gpt2m. Se desactiva xet porque las descargas se colgaban.

In [ ]:
%env HF_HUB_DISABLE_XET=1
!python -m src.data

## 2. E0: gpt2m (cross-entropy en val)

In [ ]:
!python -m src.evaluate --model gpt2m --dtype fp16

## 3. Latencia en la T4 (batch 1, sin decoder)
Se mide el `next_frame` de cada modelo; el decoder va aparte. Los alumnos son aleatorios porque la latencia no depende de los pesos.

In [ ]:
!python -m src.bench --model gpt2m --dtype fp16
!python -m src.bench --model random:gpt_s --dtype fp16 --graphs
!python -m src.bench --model random:convlstm --dtype fp16

In [ ]:
for s in (1, 2, 4, 8):
  !python -m src.bench --model random:maskgit_s --dtype fp16 --graphs --steps {s}

In [ ]:
!python -m src.bench --decoder --dtype fp16

### Inferencia de punta a punta en la T4
Video → encoder VQ → world model → decoder VQ → video, todo en la GPU (fp16 + CUDA graphs), batch 1. Ver `Docs/plan-inferencia.md`.

1. El codec de comma solo: latencia del encoder, del decoder y de los dos juntos, y fidelidad de fp16 contra fp32.
2. La cadena completa con cada configuración, en modo en línea y en imaginación, con modelos sin entrenar de la misma arquitectura (`RANDOM_MODELS=1`: la latencia no depende de los pesos, así que no hace falta bajar los checkpoints).
3. Algunas configuraciones a ritmo de cámara (un frame cada 50 ms).

Tarda ~30–40 min. Los resultados quedan en `results/infer/` y se copian a Drive para sumarlos al repo.

In [ ]:
!python -m src.infer.bench_codec --tag t4
!RANDOM_MODELS=1 TAG=t4 FRAMES=1000 PY=python bash scripts/e2e_all.sh bench
!RANDOM_MODELS=1 TAG=t4 FRAMES=1000 PY=python bash scripts/e2e_all.sh paced
!python -m src.infer.report_e2e && cat results/infer/summary_e2e.md
!mkdir -p /content/drive/MyDrive/ml2/results && cp -r results/infer /content/drive/MyDrive/ml2/results/

## 4. Entrenamiento
Los checkpoints van a Drive (`--ckpt-dir`) y la corrida se reanuda sola desde `last.pt` si Colab corta la sesión. Descomentar lo que se quiera entrenar.

In [ ]:
# !python -m src.train --kind gpt --size s --run gpt_s --tokens 100e6 --ckpt-dir {CK}
# !python -m src.train --kind gpt --size s --run gpt_s_kd --tokens 100e6 --kd --ckpt-dir {CK}
# !python -m src.train --kind maskgit --size s --run maskgit_s --tokens 100e6 --ckpt-dir {CK}
# !python -m src.train --kind convlstm --run convlstm --tokens 100e6 --ckpt-dir {CK}

## 5. Rollouts y métricas de video
Primero el set fijo de clips (400 clips), después los rollouts de cada modelo y LPIPS/FVD contra los tokens reales decodificados.

In [ ]:
# !python -m src.rollouts --make-clips
# !python -m src.rollouts --model real
# !python -m src.metrics_video --name real
# !python -m src.rollouts --model gpt2m --dtype fp16
# !python -m src.metrics_video --name gpt2m_fp16
# !python -m src.rollouts --model {CK}/maskgit_s/final.pt --steps 4
# !python -m src.metrics_video --name maskgit_s_s4

## 6. Reporte
Lee `results/configs.json` (el manifiesto lo arma el lead al final) y escribe figuras y tabla resumen.

In [ ]:
# !python -m src.report
# !cp -r results /content/drive/MyDrive/ml2/results  # copia de resultados a Drive